In [ ]:
import numpy as np
import pandas as pd

In [ ]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

amst_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv")

lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

lnd_final_beta = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv")

In [ ]:
def get_coef(city_beta, name):
    row = city_beta.loc[city_beta["Name"] == name]
    return row["Value"].values[0] if not row.empty else 0.0

def predict_mode_choice(city_beta, travel_time_min, distance_km,
                        income_quintile, age_band, gender, is_peak=0):
    g = lambda name: get_coef(city_beta, name)

    v_pt = (g("asc_pt")
            + g("b_time_pt")    * travel_time_min
            + g("b_income_pt")  * income_quintile
            + g("b_age_pt")     * age_band
            + g("b_gender_pt")  * gender
            + g("b_peak_pt")    * is_peak)

    v_bike = (g("asc_bike")
              + g("b_time_bike")   * travel_time_min
              + g("b_income_bike") * income_quintile
              + g("b_age_bike")    * age_band
              + g("b_gender_bike") * gender
              + g("b_dist_bike")   * distance_km
              + g("b_peak_bike")   * is_peak)

    exp_v = np.exp([0.0, v_pt, v_bike])
    probs = exp_v / exp_v.sum()
    return dict(zip(("car", "pt", "bike"), probs))

def predict_row(row, city_beta, dist_multiplier=1.0):
    return predict_mode_choice(
        city_beta,
        travel_time_min  = row["travel_time_min"],
        distance_km      = row["distance_km"] * dist_multiplier,
        income_quintile  = row["income_quintile"],
        age_band         = row["age_band"],
        gender           = row["gender"],
        is_peak          = row["is_peak"],
    )

def validate_model(city_beta, city_workset, n_sample=100):
    sample = city_workset.sample(n=min(n_sample, len(city_workset)), random_state=42)

    records = []
    for _, row in sample.iterrows():
        probs     = predict_row(row, city_beta)
        predicted = max(probs, key=probs.get)
        actual    = row["chosen_mode"]
        records.append({
            "actual":    actual,
            "predicted": predicted,
            **{f"p_{m}": probs[m] for m in ("car", "pt", "bike")},
            "correct":   predicted == actual,
        })

    results          = pd.DataFrame(records)
    hit_rate         = results["correct"].mean()
    mean_prob_chosen = results.apply(lambda r: r[f"p_{r['actual']}"], axis=1).mean()
    return {"hit_rate": hit_rate, "mean_prob_chosen": mean_prob_chosen, "predictions": results}

def print_validation(city_name, val):
    preds  = val["predictions"]
    cm     = pd.crosstab(preds["predicted"], preds["actual"])
    modes  = cm.columns.tolist()
    col_w  = 10

    print(f"\n{city_name}")
    print(f"  Sample        {len(preds):,}")
    print(f"  Hit rate      {val['hit_rate']:.1%}")
    print(f"  Mean P(chosen){val['mean_prob_chosen']:>7.3f}")

    print(f"\n  Predicted \\ Actual" + "".join(f"{m:>{col_w}}" for m in modes))
    for pred in cm.index:
        print(f"  {pred:<18}" + "".join(f"{cm.at[pred, m] if m in cm.columns else 0:>{col_w}}" for m in modes))

In [ ]:
print_validation("AMSTERDAM", validate_model(amst_final_beta, amst_data,  n_sample=100))
print_validation("LONDON",    validate_model(lnd_final_beta,  lnd_data,   n_sample=200))

In [ ]:
def weighted_mean(series, weights):
    return (series * weights).sum() / weights.sum()

def apply_scenario(sample, city_beta, after_fn):
    sample = sample.copy()
    sample["pred_before"] = sample.apply(lambda r: predict_row(r, city_beta), axis=1)
    sample["pred_after"]  = sample.apply(lambda r: after_fn(r, city_beta),    axis=1)
    for period in ("before", "after"):
        for mode in ("car", "pt", "bike"):
            sample[f"p_{mode}_{period}"] = sample[f"pred_{period}"].apply(lambda x: x[mode])
    return sample

def mode_shares(df, period):
    w = df["weight_trip"]
    return {m: weighted_mean(df[f"p_{m}_{period}"], w) for m in ("car", "pt", "bike")}

def print_scenario(title, sample):
    before = mode_shares(sample, "before")
    after  = mode_shares(sample, "after")
    shifts = {m: after[m] - before[m] for m in ("car", "pt", "bike")}

    print(f"\n{title}")
    print(f"  Aggregate Mode Shares  (n={len(sample):,})")
    print(f"  {'Mode':<8}{'Before':>8}{'After':>8}{'Shift':>8}")
    for m in ("car", "pt", "bike"):
        print(f"  {m.capitalize():<8}{before[m]:>8.1%}{after[m]:>8.1%}{shifts[m]:>+8.1%}")

    print(f"\n  Equity Impact by Income Quintile")
    print(f"  {'Quintile':<14}{'Car':>10}{'PT':>10}{'Bike':>10}")
    for q in range(1, 6):
        qs = sample[sample["income_quintile"] == q]
        if qs.empty:
            continue
        qb = mode_shares(qs, "before")
        qa = mode_shares(qs, "after")
        qs = {m: qa[m] - qb[m] for m in ("car", "pt", "bike")}
        print(f"  Q{q} (n={len(sample[sample['income_quintile']==q]):3d})     "
              f"{qs['car']:>+8.2%}  {qs['pt']:>+8.2%}  {qs['bike']:>+8.2%}")

In [ ]:
# Scenario 1 — London: Cycle Infrastructure (20% distance reduction)
np.random.seed(42)
lnd_sample = lnd_data.sample(n=min(1000, len(lnd_data)), random_state=42)
print_scenario(
    "SCENARIO 1  London Cycle Infrastructure  (20% distance reduction)",
    apply_scenario(lnd_sample, lnd_final_beta,
                   after_fn=lambda r, b: predict_row(r, b, dist_multiplier=0.8))
)

In [ ]:
# Scenario 2 — Amsterdam: Congestion Charge (€5/day)
def congestion_charge(row, city_beta):
    baseline = predict_row(row, city_beta)
    if baseline["car"] > 0.5:
        return predict_mode_choice(
            city_beta,
            travel_time_min = row["travel_time_min"],
            distance_km     = row["distance_km"],
            income_quintile = max(1, row["income_quintile"] - 0.5),
            age_band        = row["age_band"],
            gender          = row["gender"],
            is_peak         = row["is_peak"],
        )
    return baseline

ams_sample = amst_data.sample(n=min(500, len(amst_data)), random_state=42)
print_scenario(
    "SCENARIO 2  Amsterdam Congestion Charge  (€5/day)",
    apply_scenario(ams_sample, amst_final_beta, after_fn=congestion_charge)
)